# Import Libraries

In [1]:
import sys
sys.path.insert(0, '/home/students/acct1002_02/anaconda3/envs/tf_gpu/lib/python3.8/site-packages')

In [2]:
#Tensorflow version 2.2 for all models set-up!!
import tensorflow as tf
tf.__version__

'2.2.0'

In [3]:
import os
import json
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.applications import VGG16, ResNet50, DenseNet121
from tensorflow.keras.layers import Input, Conv2D,Dense,Flatten,Dropout,MaxPooling2D, Activation, BatchNormalization, GlobalAveragePooling2D


In [4]:
#Use CPU
os.environ["CUDA_VISIBLE_DEVICES"]="-1"

# ResNet50

In [6]:

R_model = ResNet50(weights = 'imagenet', include_top = False, input_tensor=Input(shape=(112, 112, 3)))
R_model.summary()

94773248/94765736 [==============================] - 2s 0us/step
Model: "resnet50"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 112, 112, 3) 0                                            
__________________________________________________________________________________________________
conv1_pad (ZeroPadding2D)       (None, 118, 118, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
conv1_conv (Conv2D)             (None, 56, 56, 64)   9472        conv1_pad[0][0]                  
__________________________________________________________________________________________________
conv1_bn (BatchNormalization)   (None, 56, 56, 64)   256         conv1_conv[0][0]                 
__________________________

In [8]:
last_layer_r = R_model.get_layer('conv5_block3_out').output
avg_pool = GlobalAveragePooling2D()(last_layer_r)
out = Dense(15, activation='softmax', name = 'out_put_layer')(avg_pool)
cR_model = Model(inputs = R_model.input, outputs = out)

In [9]:
for layer in cR_model.layers[:-1]:
    layer.trainable = False
cR_model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 112, 112, 3) 0                                            
__________________________________________________________________________________________________
conv1_pad (ZeroPadding2D)       (None, 118, 118, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
conv1_conv (Conv2D)             (None, 56, 56, 64)   9472        conv1_pad[0][0]                  
__________________________________________________________________________________________________
conv1_bn (BatchNormalization)   (None, 56, 56, 64)   256         conv1_conv[0][0]                 
______________________________________________________________________________________________

In [14]:
cR_model.save_weights('R_weights.h5')
json_config = cR_model.to_json()
with open('R.json', 'w') as json_file:
    json_file.write(json_config)

# DenseNet121

In [5]:
D_model = DenseNet121(weights = 'imagenet', include_top = False, input_tensor=Input(shape=(112, 112, 3)))
D_model.summary()

Model: "densenet121"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 112, 112, 3) 0                                            
__________________________________________________________________________________________________
zero_padding2d (ZeroPadding2D)  (None, 118, 118, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
conv1/conv (Conv2D)             (None, 56, 56, 64)   9408        zero_padding2d[0][0]             
__________________________________________________________________________________________________
conv1/bn (BatchNormalization)   (None, 56, 56, 64)   256         conv1/conv[0][0]                 
________________________________________________________________________________________

In [6]:
last_layer_d = D_model.get_layer('relu').output
avg_pool_d = GlobalAveragePooling2D()(last_layer_d)
out_d = Dense(15, activation='softmax', name = 'out_put_layer')(avg_pool_d)
cD_model = Model(inputs = D_model.input, outputs = out_d)

In [7]:
for layer in cD_model.layers[:-1]:
    layer.trainable = False
cD_model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 112, 112, 3) 0                                            
__________________________________________________________________________________________________
zero_padding2d (ZeroPadding2D)  (None, 118, 118, 3)  0           input_1[0][0]                    
__________________________________________________________________________________________________
conv1/conv (Conv2D)             (None, 56, 56, 64)   9408        zero_padding2d[0][0]             
__________________________________________________________________________________________________
conv1/bn (BatchNormalization)   (None, 56, 56, 64)   256         conv1/conv[0][0]                 
______________________________________________________________________________________________

In [8]:
cD_model.save_weights('D_weights.h5')
json_config = cD_model.to_json()
with open('D.json', 'w') as json_file:
    json_file.write(json_config)

# VGG16

In [13]:
VGG16_model = VGG16(weights = 'imagenet', include_top = False, input_tensor=Input(shape=(112, 112, 3)))
last_out = VGG16_model.get_layer('block5_pool').output
flt = Flatten()(last_out)
final = Dense(15, activation='softmax', name = 'out_put_layer')(flt)

VGG16_model = Model(inputs=VGG16_model.input, outputs=final)
VGG16_model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 112, 112, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 112, 112, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 112, 112, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 56, 56, 64)        0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 56, 56, 128)       73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 56, 56, 128)       147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, 28, 28, 128)       0     

In [14]:
for layer in VGG16_model.layers[:-2]:
    layer.trainable = False
VGG16_model.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_2 (InputLayer)         [(None, 112, 112, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 112, 112, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 112, 112, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 56, 56, 64)        0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 56, 56, 128)       73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 56, 56, 128)       147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, 28, 28, 128)       0     

In [15]:
VGG16_model.save_weights('VGG16_112_weights.h5')
json_config = VGG16_model.to_json()
with open('VGG16_112.json', 'w') as json_file:
    json_file.write(json_config)